
# Road Accident and Vehicle Data Cleaning

## Overview

This notebook cleans and prepares the Accident and Vehicle datasets based on the data quality issues identified in `01_data_profiling.ipynb`.

The cleaning process includes:

- Handling missing and invalid values.
- Correcting inconsistent categorical codes.
- Standardising data formats and categories.
- Checking the cleaned datasets before export.

The goal is to prepare consistent datasets for further analysis and Power BI visualisation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load datasets

accident_df = pd.read_csv("../data/raw/accident.csv", low_memory=False)
vehicle_df = pd.read_csv("../data/raw/vehicle.csv", low_memory=False)
accident_clean = accident_df.copy()
vehicle_clean = vehicle_df.copy()
print("Shape:", accident_clean.shape)
print("Shape:", vehicle_clean.shape)

Shape: (199699, 23)
Shape: (364297, 37)


## 1. Accident Data Cleaning


### 1.1. Severity Consistency Flag

Nine accidents were found to have inconsistent severity and casualty counts during data profiling.

Since the correct values cannot be confirmed, the original values are kept and a `SEVERITY_MISMATCH_FLAG` is added to identify these records.

In [2]:
severity_mismatch = accident_df[
    # Severity 1: Fatal accident must have at least one killed person
    ((accident_df['SEVERITY'] == 1) &
     (accident_df['NO_PERSONS_KILLED'] == 0))

    |

    # Severity 2: No fatalities, but at least one serious injury
    ((accident_df['SEVERITY'] == 2) &
     (
         (accident_df['NO_PERSONS_KILLED'] > 0) |
         (accident_df['NO_PERSONS_INJ_2'] == 0)
     ))

    |

    # Severity 3: No fatal/serious injuries, but at least one other injury
    ((accident_df['SEVERITY'] == 3) &
     (
         (accident_df['NO_PERSONS_KILLED'] > 0) |
         (accident_df['NO_PERSONS_INJ_2'] > 0) |
         (accident_df['NO_PERSONS_INJ_3'] == 0)
     ))

    |

    # Severity 4: No recorded injuries or fatalities
    ((accident_df['SEVERITY'] == 4) &
     (
         (accident_df['NO_PERSONS_KILLED'] > 0) |
         (accident_df['NO_PERSONS_INJ_2'] > 0) |
         (accident_df['NO_PERSONS_INJ_3'] > 0)
     ))
]

In [3]:
accident_clean['SEVERITY_MISMATCH_FLAG'] = (
    accident_clean['ACCIDENT_NO']
    .isin(severity_mismatch['ACCIDENT_NO'])
)

print(
    accident_clean['SEVERITY_MISMATCH_FLAG']
    .value_counts()
)
accident_clean[accident_clean['SEVERITY_MISMATCH_FLAG'] == 1]

SEVERITY_MISMATCH_FLAG
False    199690
True          9
Name: count, dtype: int64


,ACCIDENT_NO,ACCIDENT_DATE,ACCIDENT_TIME,ACCIDENT_TYPE,ACCIDENT_TYPE_DESC,DAY_OF_WEEK,DAY_WEEK_DESC,DCA_CODE,DCA_DESC,LIGHT_CONDITION,...,NO_PERSONS_INJ_3,NO_PERSONS_NOT_INJ,NO_PERSONS,POLICE_ATTEND,ROAD_GEOMETRY,ROAD_GEOMETRY_DESC,SEVERITY,SPEED_ZONE,RMA,SEVERITY_MISMATCH_FLAG
27975,T20210002562,2021-02-04,16:30:00,2,Struck Pedestrian,5,Thursday,109,ANY MANOEUVRE INVOLVING PED NOT INCLUDED IN DC...,1,...,0,0,1,1,5,Not at intersection,3,999,Local Road,True
39844,T20210010821,2021-05-13,20:25:00,1,Collision with vehicle,5,Thursday,111,RIGHT FAR (INTERSECTIONS ONLY),3,...,0,3,4,1,1,Cross intersection,3,40,Arterial Other,True
81879,T20210025034,2021-12-07,16:00:00,1,Collision with vehicle,3,Tuesday,130,REAR END(VEHICLES IN SAME LANE),1,...,0,4,4,1,5,Not at intersection,3,100,Arterial Highway,True
82403,T20210014197,2021-05-31,21:00:00,1,Collision with vehicle,2,Monday,160,VEHICLE COLLIDES WITH VEHICLE PARKED ON LEFT O...,6,...,0,0,1,2,5,Not at intersection,3,50,Local Road,True
97374,T20230029194,2023-12-10,07:45:00,7,Fall from or in moving vehicle,0,Sunday,190,FELL IN/FROM VEHICLE,1,...,0,2,2,1,5,Not at intersection,3,60,Arterial Other,True
108249,T20240007400,2024-03-23,17:34:00,1,Collision with vehicle,6,Saturday,110,CROSS TRAFFIC(INTERSECTIONS ONLY),1,...,1,1,2,1,1,Cross intersection,2,50,Local Road,True
171062,T20230001286,2023-01-18,11:55:00,1,Collision with vehicle,3,Wednesday,130,REAR END(VEHICLES IN SAME LANE),9,...,0,2,2,1,5,Not at intersection,3,999,Arterial Other,True
190105,T20230010484,2023-05-04,20:33:00,1,Collision with vehicle,4,Thursday,140,U TURN,3,...,0,1,1,1,2,T intersection,3,60,Arterial Highway,True
199463,T20200003573,2020-02-15,09:43:00,1,Collision with vehicle,6,Saturday,134,LANE CHANGE RIGHT (NOT OVERTAKING),1,...,0,3,3,1,5,Not at intersection,3,80,Arterial Other,True



### 1.2. Correct Weekday Values

Some `DAY_OF_WEEK` codes and weekday descriptions were inconsistent in the original dataset.

Both fields are recalculated from `ACCIDENT_DATE` using the documented weekday codes (Sunday = 1 to Saturday = 7).


In [4]:
# Convert ACCIDENT_DATE to datetime.
accident_clean['ACCIDENT_DATE'] = pd.to_datetime(
    accident_clean['ACCIDENT_DATE'],
    format='%Y-%m-%d',
    errors='coerce'
)

# Derive the correct weekday name from ACCIDENT_DATE.
accident_clean['DAY_WEEK_DESC'] = (
    accident_clean['ACCIDENT_DATE'].dt.day_name()
)

# Map weekday names back to the documented numerical codes.
weekday_code_mapping = {
    'Sunday': 1,
    'Monday': 2,
    'Tuesday': 3,
    'Wednesday': 4,
    'Thursday': 5,
    'Friday': 6,
    'Saturday': 7
}

accident_clean['DAY_OF_WEEK'] = (
    accident_clean['DAY_WEEK_DESC']
    .map(weekday_code_mapping)
)

In [5]:
# Confirm the cleaned day fields are consistent.
(
    accident_clean['DAY_OF_WEEK']
    .map({
        1: 'Sunday',
        2: 'Monday',
        3: 'Tuesday',
        4: 'Wednesday',
        5: 'Thursday',
        6: 'Friday',
        7: 'Saturday'
    })
    != accident_clean['DAY_WEEK_DESC']
).sum()

np.int64(0)

In [6]:
# Check mapping between DAY_OF_WEEK and DAY_WEEK_DESC
weekday_check = pd.crosstab(
    accident_clean['DAY_OF_WEEK'],
    accident_clean['DAY_WEEK_DESC']
)

display(weekday_check)

DAY_WEEK_DESC,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
DAY_OF_WEEK,,,,,,,
1,0,0,0,24186,0,0,0
2,0,27174,0,0,0,0,0
3,0,0,0,0,0,28912,0
4,0,0,0,0,0,0,29894
5,0,0,0,0,30219,0,0
6,31449,0,0,0,0,0,0
7,0,0,27865,0,0,0,0



### 1.3. Create Speed Zone Description

Create `SPEED_ZONE_DESC` from `SPEED_ZONE` to make speed limits easier to understand in the analysis and Power BI dashboard.

Special codes (`777`, `888`, and `999`) are kept as separate categories.


In [7]:
speed_zone_mapping = { 30: '30 km/hr',  40: '40 km/hr', 50: '50 km/hr', 60: '60 km/hr', 70: '70 km/hr',
                       75: '75 km/hr', 80: '80 km/hr', 90: '90 km/hr', 100: '100 km/hr', 110: '110 km/hr',
                       777: 'other speed limit', 888: 'Camping ground, off road', 999: 'Not known'}
accident_clean['SPEED_ZONE_DESC'] = (accident_clean['SPEED_ZONE'].map(speed_zone_mapping))
print(accident_clean[['SPEED_ZONE', 'SPEED_ZONE_DESC']].value_counts().sort_index())

SPEED_ZONE  SPEED_ZONE_DESC         
30          30 km/hr                      418
40          40 km/hr                    12891
50          50 km/hr                    32869
60          60 km/hr                    65109
70          70 km/hr                    12507
75          75 km/hr                       20
80          80 km/hr                    30291
90          90 km/hr                      503
100         100 km/hr                   28115
110         110 km/hr                    2075
777         other speed limit             392
888         Camping ground, off road     1334
999         Not known                   13175
Name: count, dtype: int64



### 1.4. Handle Missing Values

Missing `RMA` values are replaced with `UNKNOWN` to retain all accident records without assuming the missing road management category.


In [8]:

# Count missing RMA values before cleaning
missing_rma = accident_clean["RMA"].isna().sum()

# Replace missing values with an explicit unknown category
accident_clean["RMA"] = accident_clean["RMA"].fillna("UNKNOWN")

# Validate the result
print(f"Missing RMA values replaced: {missing_rma:,}")
print(f"Remaining missing values: {accident_clean['RMA'].isna().sum()}")


Missing RMA values replaced: 7,673
Remaining missing values: 0



### 1.5. Standardise Date and Time Formats

`ACCIDENT_DATE` was converted to datetime in Section 1.2. `ACCIDENT_TIME` is standardised to `HH:MM:SS`, with invalid time values converted to missing.


In [9]:

# Standardise accident time to HH:MM:SS
accident_clean["ACCIDENT_TIME"] = (
    pd.to_datetime(
        accident_clean["ACCIDENT_TIME"],
        format="%H:%M:%S",
        errors="coerce"
    )
    .dt.strftime("%H:%M:%S")
)

# Validate date and time values
print(accident_clean[["ACCIDENT_DATE", "ACCIDENT_TIME"]].dtypes)

print(
    "Invalid ACCIDENT_DATE:",
    accident_clean["ACCIDENT_DATE"].isna().sum()
)

print(
    "Invalid ACCIDENT_TIME:",
    accident_clean["ACCIDENT_TIME"].isna().sum()
)


ACCIDENT_DATE    datetime64[us]
ACCIDENT_TIME               str
dtype: object
Invalid ACCIDENT_DATE: 0
Invalid ACCIDENT_TIME: 0



### 1.6. Standardise Text Values

Remove leading and trailing whitespace from text fields to ensure consistent category values, without changing their original meaning.


In [10]:
# Standardise text values by removing leading/trailing whitespace.
text_cols = accident_clean.select_dtypes(
    include=['str']
).columns

for col in text_cols:
    accident_clean[col] = accident_clean[col].str.strip()

In [11]:
for col in [
    'ACCIDENT_TYPE_DESC',
    'DAY_WEEK_DESC',
    'DCA_DESC',
    'ROAD_GEOMETRY_DESC',
    'RMA'
]:
    print(f'\n{col}')
    print(sorted(accident_clean[col].dropna().unique())[:20])


ACCIDENT_TYPE_DESC
['Collision with a fixed object', 'Collision with vehicle', 'Fall from or in moving vehicle', 'No collision and no object struck', 'Other accident', 'Struck Pedestrian', 'Struck animal', 'Vehicle overturned (no collision)', 'collision with some other object']

DAY_WEEK_DESC
['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

DCA_DESC
['ACCIDENT OR BROKEN DOWN', 'ANY MANOEUVRE INVOLVING PED NOT INCLUDED IN DCAs 100-108.', 'CROSS TRAFFIC(INTERSECTIONS ONLY)', 'CUTTING IN (OVERTAKING)', 'DOUBLE PARKED', 'ENTERING PARKING', 'FAR SIDE. PED HIT BY VEHICLE FROM THE LEFT', 'FELL IN/FROM VEHICLE', 'HEAD ON (NOT OVERTAKING)', 'HEAD ON(OVERTAKING)', 'LANE CHANGE LEFT (NOT OVERTAKING)', 'LANE CHANGE RIGHT (NOT OVERTAKING)', 'LANE SIDE SWIPE (VEHICLES IN PARALLEL LANES)', 'LEAVING PARKING', 'LEFT FAR (INTERSECTIONS ONLY)', 'LEFT NEAR (INTERSECTIONS ONLY)', 'LEFT OFF CARRIAGEWAY INTO OBJECT/PARKED VEHICLE', 'LEFT REAR', 'LEFT THROUGH', 'LEFT TURN SIDES

## 2. Vehicle Data Cleaning


### 2.1. Flag Suspicious and Missing Values

Suspicious categorical and numerical values are flagged based on the issues identified during data profiling. Numerical thresholds are applied by vehicle type where appropriate.

Original missing values are also flagged before cleaning. Manufacture years later than the corresponding accident years are identified separately to avoid automatically treating all such records as invalid.


In [12]:
# Suspicious / undocumented categorical values
vehicle_clean['FUEL_TYPE_SUSPICIOUS_FLAG'] = (
    vehicle_clean['FUEL_TYPE'].isin(['O', 'S'])
)

vehicle_clean['TOWED_AWAY_SUSPICIOUS_FLAG'] = (
    vehicle_clean['TOWED_AWAY_FLAG'] == 9
)

In [13]:

# Vehicle types subject to light-vehicle thresholds
small_vehicle_codes = [1, 2, 3, 4, 5, 71]

# Flag invalid manufacture years
vehicle_clean["YEAR_SUSPICIOUS_FLAG"] = (
    (vehicle_clean["VEHICLE_YEAR_MANUF"] == 0)
    | (vehicle_clean["VEHICLE_YEAR_MANUF"] > 2026)
)

# Flag unusually high cylinder counts in light vehicles
vehicle_clean["CYLINDER_SUSPICIOUS_FLAG"] = (
    vehicle_clean["VEHICLE_TYPE"].isin(small_vehicle_codes)
    & (vehicle_clean["NO_OF_CYLINDERS"] > 20)
)

# Flag unusually high wheel counts in light vehicles
vehicle_clean["WHEEL_SUSPICIOUS_FLAG"] = (
    vehicle_clean["VEHICLE_TYPE"].isin(small_vehicle_codes)
    & (vehicle_clean["NO_OF_WHEELS"] > 12)
)

# Flag unusually high seating capacity in light vehicles
vehicle_clean["SEATING_SUSPICIOUS_FLAG"] = (
    vehicle_clean["VEHICLE_TYPE"].isin(small_vehicle_codes)
    & (vehicle_clean["SEATING_CAPACITY"] > 20)
)

# Flag suspicious tare weights
vehicle_clean["TARE_WEIGHT_SUSPICIOUS_FLAG"] = (
    (vehicle_clean["TARE_WEIGHT"] == 0)
    | (
        vehicle_clean["VEHICLE_TYPE"].isin(small_vehicle_codes)
        & (vehicle_clean["TARE_WEIGHT"] > 20000)
    )
)


In [14]:

# Convert accident dates to datetime before extracting the year
accident_dates = pd.to_datetime(
    accident_clean["ACCIDENT_DATE"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Create a lookup from accident ID to accident year
accident_year_lookup = pd.Series(
    accident_dates.dt.year.to_numpy(),
    index=accident_clean["ACCIDENT_NO"]
)

# Match each vehicle with its accident year
vehicle_accident_year = (
    vehicle_clean["ACCIDENT_NO"].map(accident_year_lookup)
)

# Flag manufacture years later than the accident year
vehicle_clean["YEAR_AFTER_ACCIDENT_FLAG"] = (
    vehicle_clean["VEHICLE_YEAR_MANUF"] > vehicle_accident_year
)

print(
    "Vehicles manufactured after accident year:",
    vehicle_clean["YEAR_AFTER_ACCIDENT_FLAG"].sum()
)


Vehicles manufactured after accident year: 7


In [15]:
# Flag original missing values
impute_cols = [
    'VEHICLE_YEAR_MANUF',
    'NO_OF_WHEELS',
    'NO_OF_CYLINDERS',
    'SEATING_CAPACITY',
    'TARE_WEIGHT',
    'TOTAL_NO_OCCUPANTS'
]

for col in impute_cols:
    vehicle_clean[f'{col}_WAS_MISSING'] = vehicle_clean[col].isna()

In [16]:

# Summarise suspicious and originally missing values
flag_cols = [
    col for col in vehicle_clean.columns
    if col.endswith("_SUSPICIOUS_FLAG")
    or col.endswith("_WAS_MISSING")
    or col == "YEAR_AFTER_ACCIDENT_FLAG"
]

flag_summary = (
    vehicle_clean[flag_cols]
    .sum()
    .sort_values(ascending=False)
    .to_frame("Flagged Records")
)

display(flag_summary)


,Flagged Records
SEATING_CAPACITY_WAS_MISSING,62818
TARE_WEIGHT_SUSPICIOUS_FLAG,51672
NO_OF_WHEELS_WAS_MISSING,45697
NO_OF_CYLINDERS_WAS_MISSING,38080
YEAR_SUSPICIOUS_FLAG,31077
TOWED_AWAY_SUSPICIOUS_FLAG,26525
TARE_WEIGHT_WAS_MISSING,11967
VEHICLE_YEAR_MANUF_WAS_MISSING,6951
FUEL_TYPE_SUSPICIOUS_FLAG,51
TOTAL_NO_OCCUPANTS_WAS_MISSING,31



### 2.2. Handle Outliers and Incorrect Values

Numerical values identified as implausible in Section 2.1 are replaced with missing values (`NaN`) for subsequent imputation.


In [17]:
vehicle_clean.loc[
    vehicle_clean['YEAR_SUSPICIOUS_FLAG'],
    'VEHICLE_YEAR_MANUF'
] = np.nan

vehicle_clean.loc[
    vehicle_clean['CYLINDER_SUSPICIOUS_FLAG'],
    'NO_OF_CYLINDERS'
] = np.nan

vehicle_clean.loc[
    vehicle_clean['WHEEL_SUSPICIOUS_FLAG'],
    'NO_OF_WHEELS'
] = np.nan

vehicle_clean.loc[
    vehicle_clean['SEATING_SUSPICIOUS_FLAG'],
    'SEATING_CAPACITY'
] = np.nan

vehicle_clean.loc[
    vehicle_clean['TARE_WEIGHT_SUSPICIOUS_FLAG'],
    'TARE_WEIGHT'
] = np.nan

In [18]:
groups = {
    'VEHICLE_YEAR_MANUF': 'YEAR_SUSPICIOUS_FLAG',
    'NO_OF_CYLINDERS': 'CYLINDER_SUSPICIOUS_FLAG',
    'NO_OF_WHEELS': 'WHEEL_SUSPICIOUS_FLAG',
    'SEATING_CAPACITY': 'SEATING_SUSPICIOUS_FLAG',
    'TARE_WEIGHT': 'TARE_WEIGHT_SUSPICIOUS_FLAG'
}

for value_col, flag_col in groups.items():
    print(f"\n=== {value_col} ===")

    display(
        vehicle_clean.loc[
            vehicle_clean[flag_col],
            ['ACCIDENT_NO', 'VEHICLE_ID', value_col, flag_col]
        ].head(5)
    )


=== VEHICLE_YEAR_MANUF ===


,ACCIDENT_NO,VEHICLE_ID,VEHICLE_YEAR_MANUF,YEAR_SUSPICIOUS_FLAG
55,T20130024566,B,NaN,True
62,T20140001259,A,NaN,True
92,T20140019016,B,NaN,True
95,T20140021737,B,NaN,True
103,T20150001409,A,NaN,True



=== NO_OF_CYLINDERS ===


,ACCIDENT_NO,VEHICLE_ID,NO_OF_CYLINDERS,CYLINDER_SUSPICIOUS_FLAG
9660,T20140005298,A,NaN,True
19512,T20140003721,B,NaN,True
31677,T20120017654,A,NaN,True
122953,T20130023255,B,NaN,True
152032,T20250007311,A,NaN,True



=== NO_OF_WHEELS ===


,ACCIDENT_NO,VEHICLE_ID,NO_OF_WHEELS,WHEEL_SUSPICIOUS_FLAG
153743,T20120023815,A,NaN,True
255047,T20250019924,A,NaN,True



=== SEATING_CAPACITY ===


,ACCIDENT_NO,VEHICLE_ID,SEATING_CAPACITY,SEATING_SUSPICIOUS_FLAG
73096,T20220019246,A,NaN,True
77591,T20190014606,B,NaN,True
91074,T20180005970,B,NaN,True
122395,T20220025163,A,NaN,True
161036,T20160013193,B,NaN,True



=== TARE_WEIGHT ===


,ACCIDENT_NO,VEHICLE_ID,TARE_WEIGHT,TARE_WEIGHT_SUSPICIOUS_FLAG
55,T20130024566,B,NaN,True
62,T20140001259,A,NaN,True
95,T20140021737,B,NaN,True
98,T20140025066,B,NaN,True
103,T20150001409,A,NaN,True



### 2.3. Standardise Unknown Values

Known unknown-value placeholders in categorical text fields are standardised to `UNKNOWN` to ensure consistent category labels.

Numerical codes are preserved and handled separately according to the data dictionary.


In [19]:
# Standardise confirmed unknown text values as missing
unknown_values = [
    'NK', 'Unknown', 'Not Known',
    'Not known', 'UNKN', 'UNK', '?'
]

vehicle_clean = vehicle_clean.replace(
    unknown_values,
    np.nan
)


### 2.4. Remove Unused Columns

`VEHICLE_POWER`, `VEHICLE_WEIGHT`, `CARRY_CAPACITY`, and `CUBIC_CAPACITY` are removed due to their high missing rates (86-100%) and limited relevance to the planned analysis.

`VEHICLE_COLOUR_2` is also removed because most records contain an unknown code (`ZZ`).


In [20]:

# Remove attributes with high missing rates or limited analytical value
drop_cols = [
    "VEHICLE_POWER",
    "VEHICLE_WEIGHT",
    "CARRY_CAPACITY",
    "CUBIC_CAPACITY",
    "VEHICLE_COLOUR_2"
]

# Record the number of columns before cleaning
cols_before = vehicle_clean.shape[1]

# Remove selected columns
vehicle_clean = vehicle_clean.drop(columns=drop_cols)

print(f"Columns removed: {cols_before - vehicle_clean.shape[1]}")
print(f"Remaining columns: {vehicle_clean.shape[1]}")


Columns removed: 5
Remaining columns: 46



### 2.5. Handle Missing Numerical Values

Missing numerical values are imputed using median values from similar vehicles.

Manufacture year and cylinder count are imputed by vehicle type and make, followed by vehicle type. Other numerical attributes are imputed by vehicle type. The overall median is used when group-level values are unavailable.

Imputed values are flagged to distinguish estimated values from the original data.


In [21]:

# Define imputation rules in priority order
imputation_rules = {
    "VEHICLE_YEAR_MANUF": [
        ["VEHICLE_TYPE_DESC", "VEHICLE_MAKE"],
        ["VEHICLE_TYPE_DESC"]
    ],
    "NO_OF_CYLINDERS": [
        ["VEHICLE_TYPE_DESC", "VEHICLE_MAKE"],
        ["VEHICLE_TYPE_DESC"]
    ],
    "NO_OF_WHEELS": [["VEHICLE_TYPE_DESC"]],
    "SEATING_CAPACITY": [["VEHICLE_TYPE_DESC"]],
    "TARE_WEIGHT": [["VEHICLE_TYPE_DESC"]],
    "TOTAL_NO_OCCUPANTS": [["VEHICLE_TYPE_DESC"]]
}

# Attributes that should contain whole numbers
round_cols = [
    "VEHICLE_YEAR_MANUF",
    "NO_OF_CYLINDERS",
    "NO_OF_WHEELS",
    "SEATING_CAPACITY",
    "TOTAL_NO_OCCUPANTS"
]

imputation_summary = []

for col, group_levels in imputation_rules.items():

    # Identify missing values before imputation
    missing_mask = vehicle_clean[col].isna()

    # Preserve original non-missing values
    filled_values = vehicle_clean[col].copy()

    # Apply group-based median imputation
    for group_cols in group_levels:
        group_median = (
            vehicle_clean.groupby(group_cols)[col]
            .transform("median")
        )

        filled_values = filled_values.fillna(group_median)

    # Use overall median for any remaining missing values
    filled_values = filled_values.fillna(
        vehicle_clean[col].median()
    )

    # Round only newly imputed discrete values
    if col in round_cols:
        filled_values.loc[missing_mask] = (
            filled_values.loc[missing_mask].round()
        )

    # Flag records with imputed values
    vehicle_clean[f"{col}_WAS_IMPUTED"] = (
        missing_mask & filled_values.notna()
    )

    # Update the cleaned column
    vehicle_clean[col] = filled_values

    # Record imputation results
    imputation_summary.append({
        "Attribute": col,
        "Values Imputed": int(
            vehicle_clean[f"{col}_WAS_IMPUTED"].sum()
        ),
        "Remaining Missing": int(
            vehicle_clean[col].isna().sum()
        )
    })

# Display imputation summary
display(pd.DataFrame(imputation_summary))


,Attribute,Values Imputed,Remaining Missing
0,VEHICLE_YEAR_MANUF,38028,0
1,NO_OF_CYLINDERS,38093,0
2,NO_OF_WHEELS,45699,0
3,SEATING_CAPACITY,62825,0
4,TARE_WEIGHT,63639,0
5,TOTAL_NO_OCCUPANTS,31,0



### 2.6. Handle Missing Categorical Values

Missing categorical descriptions are replaced with `UNKNOWN`, while coded attributes use documented unknown codes where available.

For `TOWED_AWAY_FLAG` and `VEHICLE_DCA_CODE`, missing values are represented by `-1` as a project-defined placeholder. The original records are retained.


In [22]:

# Select text columns, excluding identifiers
identifier_cols = ["ACCIDENT_NO", "VEHICLE_ID"]

categorical_cols = [
    col for col in vehicle_clean.select_dtypes(
        include=["object", "string"]
    ).columns
    if col not in identifier_cols
]

# Replace missing categorical descriptions with UNKNOWN
vehicle_clean[categorical_cols] = (
    vehicle_clean[categorical_cols].fillna("UNKNOWN")
)

# Use documented unknown codes for coded attributes
unknown_code_map = {
    "ROAD_SURFACE_TYPE": 9,
    "DRIVER_INTENT": 99,
    "VEHICLE_MOVEMENT": 99,
    "CAUGHT_FIRE": 9,
    "LAMPS": 9,
    "TRAFFIC_CONTROL": 99
}

for col, code in unknown_code_map.items():
    vehicle_clean[col] = vehicle_clean[col].fillna(code)

# Use project-defined placeholders for missing coded values
placeholder_cols = [
    "TOWED_AWAY_FLAG",
    "VEHICLE_DCA_CODE"
]

# Flag missing values before replacement
for col in placeholder_cols:
    vehicle_clean[f"{col}_WAS_MISSING"] = (
        vehicle_clean[col].isna()
    )

# Replace missing values with -1
vehicle_clean[placeholder_cols] = (
    vehicle_clean[placeholder_cols].fillna(-1)
)

# Review remaining missing values
remaining_missing = (
    vehicle_clean.isna()
    .sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

print("Remaining missing values:")
display(remaining_missing)


Remaining missing values:


Series([], dtype: int64)

In [23]:
print("Missing values before imputation:")
print(vehicle_df.isna().sum())
print("Missing values after imputation:")
print(vehicle_clean.isna().sum())

Missing values before imputation:
ACCIDENT_NO                    0
VEHICLE_ID                     0
VEHICLE_YEAR_MANUF          6951
VEHICLE_DCA_CODE             925
INITIAL_DIRECTION              1
ROAD_SURFACE_TYPE              1
ROAD_SURFACE_TYPE_DESC         1
REG_STATE                  22365
VEHICLE_BODY_STYLE         28242
VEHICLE_MAKE               20044
VEHICLE_MODEL              33989
VEHICLE_POWER             364297
VEHICLE_TYPE                   0
VEHICLE_TYPE_DESC              0
VEHICLE_WEIGHT            317910
CONSTRUCTION_TYPE          71709
FUEL_TYPE                  20052
NO_OF_WHEELS               45697
NO_OF_CYLINDERS            38080
SEATING_CAPACITY           62818
TARE_WEIGHT                11967
TOTAL_NO_OCCUPANTS            31
CARRY_CAPACITY            318091
CUBIC_CAPACITY            313446
FINAL_DIRECTION                1
DRIVER_INTENT                  4
VEHICLE_MOVEMENT               7
TRAILER_TYPE                  31
VEHICLE_COLOUR_1               0
VEHICLE_C


## 3. Cross-Dataset Cleaning

Four accidents have inconsistent vehicle counts between the Accident and Vehicle datasets.

These accidents and their associated vehicle records are excluded from the cleaned datasets. The excluded records are saved separately for traceability.


In [24]:

# Identify accidents with mismatched vehicle counts
actual_vehicle_counts = (
    vehicle_clean.groupby("ACCIDENT_NO")
    .size()
)

reported_vs_actual = accident_clean[
    ["ACCIDENT_NO", "NO_OF_VEHICLES"]
].copy()

reported_vs_actual["ACTUAL_VEHICLE_COUNT"] = (
    reported_vs_actual["ACCIDENT_NO"]
    .map(actual_vehicle_counts)
    .fillna(0)
    .astype(int)
)

mismatch_ids = reported_vs_actual.loc[
    reported_vs_actual["NO_OF_VEHICLES"]
    != reported_vs_actual["ACTUAL_VEHICLE_COUNT"],
    "ACCIDENT_NO"
].tolist()

print(f"Mismatched accidents: {len(mismatch_ids)}")

# Separate exception records
accident_exception = accident_clean.loc[
    accident_clean["ACCIDENT_NO"].isin(mismatch_ids)
].copy()

vehicle_exception = vehicle_clean.loc[
    vehicle_clean["ACCIDENT_NO"].isin(mismatch_ids)
].copy()

# Exclude mismatched records from the cleaned datasets
accident_clean = accident_clean.loc[
    ~accident_clean["ACCIDENT_NO"].isin(mismatch_ids)
].copy()

vehicle_clean = vehicle_clean.loc[
    ~vehicle_clean["ACCIDENT_NO"].isin(mismatch_ids)
].copy()

# Validate the record counts
print(f"Accident exception rows: {len(accident_exception):,}")
print(f"Vehicle exception rows: {len(vehicle_exception):,}")
print(f"Cleaned Accident rows: {len(accident_clean):,}")
print(f"Cleaned Vehicle rows: {len(vehicle_clean):,}")

# Ensure excluded records are not present in cleaned datasets
assert not accident_clean["ACCIDENT_NO"].isin(mismatch_ids).any()
assert not vehicle_clean["ACCIDENT_NO"].isin(mismatch_ids).any()


Mismatched accidents: 4
Accident exception rows: 4
Vehicle exception rows: 3
Cleaned Accident rows: 199,695
Cleaned Vehicle rows: 364,294



## 4. Final Data Validation

Validate the cleaned Accident and Vehicle datasets by checking record counts, unique identifiers, missing values, and key cleaning rules.

Cross-dataset checks are also performed to confirm that the excluded vehicle count mismatches have been resolved.


### 4.1. Accident Dataset

In [25]:

# Check record counts
print("Original Accident records:", len(accident_df))
print("Exception records:", len(accident_exception))
print("Cleaned Accident records:", len(accident_clean))

# Validate record preservation
assert len(accident_df) == len(accident_clean) + len(accident_exception)

# Check identifiers and duplicates
assert accident_clean["ACCIDENT_NO"].notna().all()
assert accident_clean["ACCIDENT_NO"].is_unique
assert accident_clean.duplicated().sum() == 0

# Check cleaning results
assert accident_clean["RMA"].notna().all()
assert accident_clean["DAY_OF_WEEK"].between(1, 7).all()

assert (
    accident_clean["DAY_OF_WEEK"]
    == accident_clean["DAY_WEEK_DESC"].map(weekday_code_mapping)
).all()

assert accident_clean["SPEED_ZONE_DESC"].notna().all()

# Review severity flags and missing values
print("\nSeverity mismatches retained:",
      accident_clean["SEVERITY_MISMATCH_FLAG"].sum())

print("\nRemaining missing values:")
display(
    accident_clean.isna().sum()
    .loc[lambda x: x > 0]
    .to_frame("Missing Values")
)

print("\nAccident validation completed.")


Original Accident records: 199699
Exception records: 4
Cleaned Accident records: 199695

Severity mismatches retained: 9

Remaining missing values:


,Missing Values



Accident validation completed.


### 4.2. Vehicle Dataset

In [26]:

# Check record counts
print("Original Vehicle records:", len(vehicle_df))
print("Exception records:", len(vehicle_exception))
print("Cleaned Vehicle records:", len(vehicle_clean))

# Validate record preservation
assert len(vehicle_df) == len(vehicle_clean) + len(vehicle_exception)

# Check identifiers and duplicates
assert vehicle_clean[["ACCIDENT_NO", "VEHICLE_ID"]].notna().all().all()

assert not vehicle_clean.duplicated(
    subset=["ACCIDENT_NO", "VEHICLE_ID"]
).any()

assert vehicle_clean.duplicated().sum() == 0

# Validate numerical imputation
imputed_cols = [
    "VEHICLE_YEAR_MANUF",
    "NO_OF_CYLINDERS",
    "NO_OF_WHEELS",
    "SEATING_CAPACITY",
    "TARE_WEIGHT",
    "TOTAL_NO_OCCUPANTS"
]

print("\nRemaining missing numerical values:")
display(
    vehicle_clean[imputed_cols]
    .isna().sum()
    .to_frame("Missing Values")
)

assert vehicle_clean[imputed_cols].notna().all().all()

# Confirm flagged invalid values were treated
invalid_value_rules = {
    "VEHICLE_YEAR_MANUF": "YEAR_SUSPICIOUS_FLAG",
    "NO_OF_CYLINDERS": "CYLINDER_SUSPICIOUS_FLAG",
    "NO_OF_WHEELS": "WHEEL_SUSPICIOUS_FLAG",
    "SEATING_CAPACITY": "SEATING_SUSPICIOUS_FLAG",
    "TARE_WEIGHT": "TARE_WEIGHT_SUSPICIOUS_FLAG"
}

print("\nSuspicious values replaced and imputed:")
for col, flag in invalid_value_rules.items():
    print(
        f"{col}: {vehicle_clean[flag].sum():,} flagged, "
        f"{vehicle_clean.loc[vehicle_clean[flag], f'{col}_WAS_IMPUTED'].sum():,} imputed"
    )

# Review all remaining missing values
print("\nRemaining missing values:")
display(
    vehicle_clean.isna().sum()
    .loc[lambda x: x > 0]
    .to_frame("Missing Values")
)

print("\nVehicle validation completed.")


Original Vehicle records: 364297
Exception records: 3
Cleaned Vehicle records: 364294

Remaining missing numerical values:


,Missing Values
VEHICLE_YEAR_MANUF,0
NO_OF_CYLINDERS,0
NO_OF_WHEELS,0
SEATING_CAPACITY,0
TARE_WEIGHT,0
TOTAL_NO_OCCUPANTS,0



Suspicious values replaced and imputed:
VEHICLE_YEAR_MANUF: 31,076 flagged, 31,076 imputed
NO_OF_CYLINDERS: 13 flagged, 13 imputed
NO_OF_WHEELS: 2 flagged, 2 imputed
SEATING_CAPACITY: 7 flagged, 7 imputed
TARE_WEIGHT: 51,671 flagged, 51,671 imputed

Remaining missing values:


,Missing Values



Vehicle validation completed.


### 4.3. Cross-Dataset Consistency

In [27]:

# Check vehicles without matching accident records
unmatched_vehicles = (
    ~vehicle_clean["ACCIDENT_NO"].isin(accident_clean["ACCIDENT_NO"])
).sum()

# Count actual vehicle records per accident
actual_vehicle_counts = (
    vehicle_clean.groupby("ACCIDENT_NO").size()
)

recorded_counts = (
    accident_clean["ACCIDENT_NO"]
    .map(actual_vehicle_counts)
    .fillna(0)
    .astype(int)
)

# Check vehicle count mismatches
vehicle_count_mismatches = (
    accident_clean["NO_OF_VEHICLES"] != recorded_counts
).sum()

# Check unmatched accident records
unmatched_accidents = (
    ~accident_clean["ACCIDENT_NO"].isin(vehicle_clean["ACCIDENT_NO"])
).sum()

# Confirm exceptions are excluded
exception_ids = set(accident_exception["ACCIDENT_NO"])

exceptions_remaining = (
    accident_clean["ACCIDENT_NO"].isin(exception_ids).sum()
    + vehicle_clean["ACCIDENT_NO"].isin(exception_ids).sum()
)

# Display validation summary
cross_validation_summary = pd.DataFrame({
    "Check": [
        "Vehicles without matching accidents",
        "Accidents without matching vehicles",
        "Vehicle count mismatches",
        "Exception records remaining"
    ],
    "Count": [
        unmatched_vehicles,
        unmatched_accidents,
        vehicle_count_mismatches,
        exceptions_remaining
    ]
})

display(cross_validation_summary)

# Confirm relationship consistency
assert unmatched_vehicles == 0
assert vehicle_count_mismatches == 0
assert exceptions_remaining == 0

print("Cross-dataset validation completed.")


,Check,Count
0,Vehicles without matching accidents,0
1,Accidents without matching vehicles,6
2,Vehicle count mismatches,0
3,Exception records remaining,0


Cross-dataset validation completed.



## 5. Export Cleaned Datasets

Export the cleaned Accident and Vehicle datasets for further analysis and Power BI visualisation.

Records excluded due to vehicle count mismatches are saved separately as exception files for traceability.


In [29]:
# Define output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Export cleaned datasets
accident_clean.to_csv(
    output_dir / "accident_clean.csv",
    index=False
)

vehicle_clean.to_csv(
    output_dir / "vehicle_clean.csv",
    index=False
)

# Export excluded records separately
accident_exception.to_csv(
    output_dir / "accident_exception.csv",
    index=False
)

vehicle_exception.to_csv(
    output_dir / "vehicle_exception.csv",
    index=False
)

# Confirm exported files and record counts
exported_files = {
    "accident_clean.csv": len(accident_clean),
    "vehicle_clean.csv": len(vehicle_clean),
    "accident_exception.csv": len(accident_exception),
    "vehicle_exception.csv": len(vehicle_exception)
}

for filename, count in exported_files.items():
    print(f"{filename}: {count:,} rows")


accident_clean.csv: 199,695 rows
vehicle_clean.csv: 364,294 rows
accident_exception.csv: 4 rows
vehicle_exception.csv: 3 rows
